# Robustness of CNN, Vision Transformer, and Hybrid Architectures to Image Quality Degradation in Dermatological Disease Classification

**Authors:** [Your Name], [Co-author Name]  
**Affiliation:** Independent Student Research  
**Dataset:** HAM10000 — Human Against Machine with 10000 training images (Kaggle)  
**Date:** 2025

---

## Abstract (draft — fill in after results)

Deep learning models for dermatological image classification are typically trained and evaluated on high-quality images, yet real-world clinical deployment — particularly in low-resource or rural settings — often involves degraded image quality from suboptimal lighting, low-resolution cameras, and compression artifacts. This study systematically evaluates the robustness of three architectural paradigms — Convolutional Neural Networks (EfficientNetB0), Vision Transformers (ViT-B/16), and Hybrid CNN-Transformer models (EfficientFormer) — to four types of controlled image degradation: Gaussian noise, Gaussian blur, resolution downsampling, and JPEG compression. Trained on the HAM10000 skin lesion dataset (7 classes, 10,015 images), we measure classification accuracy and F1-score degradation curves across five severity levels per corruption type. Our results reveal [FILL IN AFTER RESULTS — e.g. 'that ViT-based models maintain superior robustness under noise and blur, while CNNs degrade faster at higher severity levels, with implications for dermatoscope deployment in resource-limited settings'].

---

## 1. Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q kaggle timm scikit-learn matplotlib seaborn pandas Pillow opencv-python-headless

In [ ]:
import os
import io
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image, ImageFilter
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm  # For ViT and EfficientFormer

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)

# ── Reproducibility ─────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")

## 2. Dataset Download

**HAM10000** — Human Against Machine with 10000 training images  
- 10,015 dermatoscopic images  
- 7 classes: Melanoma, Melanocytic nevi, Basal cell carcinoma, Actinic keratoses, Benign keratosis, Dermatofibroma, Vascular lesions  
- Collected from Austria and Australia  
- Clinically significant: represents the exact setting where image quality varies widely

**Setup:** Go to kaggle.com → Account → Create API Token → download `kaggle.json`

In [ ]:
from google.colab import files

print("Upload your kaggle.json:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d data/

print("Done!")

## 3. Exploratory Data Analysis

Before any modeling, understand what we're working with. This section becomes **Section 3 (Dataset)** in your paper.

In [ ]:
# ── Load metadata ───────────────────────────────────────────────
df = pd.read_csv('data/HAM10000_metadata.csv')

# Map short codes to readable names
CLASS_NAMES = {
    'nv':   'Melanocytic Nevi',
    'mel':  'Melanoma',
    'bkl':  'Benign Keratosis',
    'bcc':  'Basal Cell Carcinoma',
    'akiec':'Actinic Keratoses',
    'vasc': 'Vascular Lesions',
    'df':   'Dermatofibroma'
}
CLASS_TO_IDX = {k: i for i, k in enumerate(CLASS_NAMES.keys())}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

df['label'] = df['dx'].map(CLASS_TO_IDX)
df['class_name'] = df['dx'].map(CLASS_NAMES)

print(f"Total images: {len(df)}")
print(f"Classes: {df['dx'].nunique()}")
print("\nClass distribution:")
print(df['dx'].value_counts())

In [ ]:
# ── Class distribution chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
counts = df['class_name'].value_counts()
colors = plt.cm.Set2(np.linspace(0, 1, len(counts)))
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.7)
ax.set_title('HAM10000 Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Disease Class')
ax.set_ylabel('Number of Images')
ax.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(val), ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
# NOTE: Strong class imbalance (nv dominates) — address with weighted loss

In [ ]:
# ── Find image paths ────────────────────────────────────────────
# HAM10000 splits images across two folders
img_dirs = [
    'data/HAM10000_images_part_1',
    'data/HAM10000_images_part_2'
]

img_path_map = {}
for d in img_dirs:
    if os.path.exists(d):
        for f in os.listdir(d):
            if f.endswith('.jpg'):
                img_id = f.replace('.jpg', '')
                img_path_map[img_id] = os.path.join(d, f)

df['path'] = df['image_id'].map(img_path_map)
df = df.dropna(subset=['path'])  # Remove any missing
print(f"Images found: {len(df)}")

In [ ]:
# ── Sample images per class ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Images — HAM10000 (one per class)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (code, name) in enumerate(CLASS_NAMES.items()):
    sample = df[df['dx'] == code].sample(1).iloc[0]
    img = Image.open(sample['path'])
    axes[i].imshow(img)
    axes[i].set_title(f'{name}\n({code})', fontsize=9, fontweight='bold')
    axes[i].axis('off')

axes[-1].axis('off')  # Hide last empty subplot
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sample_images.png  ← Figure 1 of your paper")

## 4. Image Degradation Functions

This is the **core novel contribution** of the paper. We implement 4 degradation types at 5 severity levels each. These simulate real-world clinical conditions:

| Degradation | Clinical Analog |
|---|---|
| Gaussian Noise | Sensor noise from cheap cameras |
| Gaussian Blur | Camera shake, out-of-focus dermatoscope |
| Resolution Downsampling | Low-res smartphone cameras |
| JPEG Compression | Image compression in telemedicine apps |

Severity level 0 = clean image (baseline). Levels 1-5 = increasing degradation.

In [ ]:
# ── Degradation parameters ──────────────────────────────────────
DEGRADATION_PARAMS = {
    'gaussian_noise': [0, 10, 25, 50, 75, 100],    # std of noise
    'gaussian_blur':  [0, 1,  2,  3,  5,  7],      # kernel radius
    'downsampling':   [224, 112, 56, 28, 14, 7],    # target resolution
    'jpeg_compression': [100, 80, 60, 40, 20, 5],   # JPEG quality
}

IMG_SIZE = 224


def apply_gaussian_noise(img_tensor, std):
    """Add Gaussian noise to a tensor image (values 0-1)."""
    if std == 0:
        return img_tensor
    noise = torch.randn_like(img_tensor) * (std / 255.0)
    return torch.clamp(img_tensor + noise, 0, 1)


def apply_gaussian_blur(pil_img, radius):
    """Apply Gaussian blur to a PIL image."""
    if radius == 0:
        return pil_img
    return pil_img.filter(ImageFilter.GaussianBlur(radius=radius))


def apply_downsampling(pil_img, target_size):
    """Downsample then upsample back to IMG_SIZE (information loss)."""
    if target_size == IMG_SIZE:
        return pil_img
    small = pil_img.resize((target_size, target_size), Image.BILINEAR)
    return small.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)


def apply_jpeg_compression(pil_img, quality):
    """Simulate JPEG compression at given quality (1-100)."""
    if quality == 100:
        return pil_img
    buffer = io.BytesIO()
    pil_img.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert('RGB')


print("Degradation functions defined.")

In [ ]:
# ── Visualize degradation levels ────────────────────────────────
# Pick one sample image to show all degradations
sample_path = df.sample(1).iloc[0]['path']
sample_img = Image.open(sample_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))

fig, axes = plt.subplots(4, 6, figsize=(18, 12))
fig.suptitle('Image Degradation Types × Severity Levels', fontsize=15, fontweight='bold')

degradation_fns = {
    'Gaussian Noise\n(std)': lambda img, p: apply_gaussian_noise(
        transforms.ToTensor()(img), p
    ).permute(1,2,0).numpy() if p > 0 else np.array(img)/255.0,
    'Gaussian Blur\n(radius)': lambda img, p: np.array(apply_gaussian_blur(img, p))/255.0,
    'Downsampling\n(resolution)': lambda img, p: np.array(apply_downsampling(img, p))/255.0,
    'JPEG Compression\n(quality)': lambda img, p: np.array(apply_jpeg_compression(img, p))/255.0,
}

for row, (deg_name, fn) in enumerate(degradation_fns.items()):
    params = list(DEGRADATION_PARAMS.values())[row]
    for col, param in enumerate(params):
        degraded = fn(sample_img.copy(), param)
        degraded = np.clip(degraded, 0, 1)
        axes[row][col].imshow(degraded)
        axes[row][col].set_title(f'{param}', fontsize=8)
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(deg_name, fontsize=9, fontweight='bold', rotation=90, labelpad=10)

# Label severity levels
for col, label in enumerate(['Level 0\n(Clean)', 'Level 1', 'Level 2', 'Level 3', 'Level 4', 'Level 5\n(Worst)']):
    axes[0][col].set_title(f'{label}\n{list(DEGRADATION_PARAMS.values())[0][col]}', fontsize=8)

plt.tight_layout()
plt.savefig('degradation_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: degradation_examples.png  ← Figure 2 of your paper")

## 5. Dataset Preparation

We train all models on **clean images only**, then evaluate on both clean and degraded test sets. This simulates the real scenario: models trained in ideal conditions, deployed in non-ideal conditions.

In [ ]:
from sklearn.model_selection import train_test_split

# ── Train/val/test split ────────────────────────────────────────
# Stratified so each class is represented proportionally
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── Class weights for imbalanced dataset ────────────────────────
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = torch.FloatTensor(1.0 / class_counts)
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = class_weights.to(DEVICE)
print("Class weights computed (inverse frequency weighting)")

In [ ]:
# ── PyTorch Dataset class ───────────────────────────────────────
class SkinLesionDataset(Dataset):
    """
    Loads HAM10000 images with optional degradation applied at inference time.
    
    Args:
        df: DataFrame with 'path' and 'label' columns
        transform: torchvision transforms
        degradation_type: None for clean, or one of the 4 types
        severity: 0-5 severity level
    """
    def __init__(self, df, transform=None,
                 degradation_type=None, severity=0):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.degradation_type = degradation_type
        self.severity = severity

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))

        # Apply PIL-level degradations before converting to tensor
        if self.degradation_type == 'gaussian_blur' and self.severity > 0:
            radius = DEGRADATION_PARAMS['gaussian_blur'][self.severity]
            img = apply_gaussian_blur(img, radius)

        elif self.degradation_type == 'downsampling' and self.severity > 0:
            target = DEGRADATION_PARAMS['downsampling'][self.severity]
            img = apply_downsampling(img, target)

        elif self.degradation_type == 'jpeg_compression' and self.severity > 0:
            quality = DEGRADATION_PARAMS['jpeg_compression'][self.severity]
            img = apply_jpeg_compression(img, quality)

        # Apply transforms (converts to tensor)
        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)

        # Apply tensor-level degradations
        if self.degradation_type == 'gaussian_noise' and self.severity > 0:
            std = DEGRADATION_PARAMS['gaussian_noise'][self.severity]
            img = apply_gaussian_noise(img, std)

        label = torch.tensor(row['label'], dtype=torch.long)
        return img, label


# ── Transforms ──────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Dataloaders for training (clean images) ─────────────────────
BATCH_SIZE = 32

train_loader = DataLoader(
    SkinLesionDataset(train_df, transform=train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    SkinLesionDataset(val_df, transform=eval_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 6. Model Architectures

Three architectural paradigms — representing a progression from local to global feature extraction:

| Model | Type | Key Property |
|---|---|---|
| EfficientNetB0 | Pure CNN | Local features via convolutions, inductive bias toward spatial locality |
| ViT-B/16 | Pure Transformer | Global attention from the start, no spatial inductive bias |
| EfficientFormer-L1 | Hybrid CNN+Transformer | Local CNN features early, global attention late |

In [ ]:
NUM_CLASSES = 7


def build_efficientnet():
    """Pure CNN — EfficientNetB0 with custom classification head."""
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
    return model


def build_vit():
    """Pure Vision Transformer — ViT-B/16 pretrained on ImageNet-21k."""
    model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=NUM_CLASSES)
    return model


def build_efficientformer():
    """Hybrid CNN+Transformer — EfficientFormer-L1."""
    model = timm.create_model('efficientformer_l1', pretrained=True, num_classes=NUM_CLASSES)
    return model


# Build and count parameters
model_builders = {
    'EfficientNetB0 (CNN)':       build_efficientnet,
    'ViT-B/16 (Transformer)':     build_vit,
    'EfficientFormer-L1 (Hybrid)': build_efficientformer,
}

for name, builder in model_builders.items():
    m = builder()
    params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{name:35s}: {params:.1f}M parameters")
    del m

## 7. Training Loop

All three models are trained under **identical conditions** — same optimizer, LR schedule, epochs, augmentation. This is critical for a fair comparison and is what makes the paper scientifically valid.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return total_loss / total, correct / total, f1


def train_model(name, builder, train_loader, val_loader,
                epochs=25, lr=1e-4):
    print(f"\n{'='*55}")
    print(f" Training: {name}")
    print(f"{'='*55}")

    model = builder().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
    best_val_f1 = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        v_loss, v_acc, v_f1 = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        history['val_f1'].append(v_f1)

        if v_f1 > best_val_f1:
            best_val_f1 = v_f1
            best_state = deepcopy(model.state_dict())

        if epoch % 5 == 0:
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | "
                  f"Val Loss: {v_loss:.4f} Acc: {v_acc:.4f} F1: {v_f1:.4f}")

    # Restore best weights
    model.load_state_dict(best_state)
    print(f"\nBest Val F1: {best_val_f1:.4f}")
    return model, history


print("Training functions ready.")

In [ ]:
# ── Train all three models ──────────────────────────────────────
# NOTE: This takes ~30-60 min on T4 GPU. Get a coffee.
EPOCHS = 25

trained_models = {}
histories = {}

for name, builder in model_builders.items():
    model, history = train_model(name, builder, train_loader, val_loader, epochs=EPOCHS)
    trained_models[name] = model
    histories[name] = history
    # Save checkpoint
    torch.save(model.state_dict(), f"{name.split()[0]}_best.pth")
    print(f"Saved checkpoint: {name.split()[0]}_best.pth")

In [ ]:
# ── Plot training curves ────────────────────────────────────────
fig, axes = plt.subplots(len(trained_models), 2, figsize=(14, 4*len(trained_models)))
fig.suptitle('Training Curves — All Architectures', fontsize=14, fontweight='bold')

for i, (name, history) in enumerate(histories.items()):
    epochs = range(1, len(history['train_acc']) + 1)

    axes[i][0].plot(epochs, history['train_acc'], label='Train', color='#3498db', lw=2)
    axes[i][0].plot(epochs, history['val_acc'],   label='Val',   color='#e74c3c', lw=2, ls='--')
    axes[i][0].set_title(f'{name} — Accuracy', fontweight='bold')
    axes[i][0].set_xlabel('Epoch'); axes[i][0].set_ylabel('Accuracy')
    axes[i][0].legend(); axes[i][0].grid(alpha=0.3)

    axes[i][1].plot(epochs, history['train_loss'], label='Train', color='#3498db', lw=2)
    axes[i][1].plot(epochs, history['val_loss'],   label='Val',   color='#e74c3c', lw=2, ls='--')
    axes[i][1].set_title(f'{name} — Loss', fontweight='bold')
    axes[i][1].set_xlabel('Epoch'); axes[i][1].set_ylabel('Loss')
    axes[i][1].legend(); axes[i][1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Robustness Evaluation — The Core Experiment

This is what makes the paper novel. We evaluate each trained model on:
- Clean test set (severity 0 = baseline)
- 4 degradation types × 5 severity levels = 20 degraded test sets

Total evaluations per model: **21**  
Total evaluations: **63**

We measure both accuracy AND weighted F1 (better for imbalanced classes).

In [ ]:
@torch.no_grad()
def evaluate_robustness(model, test_df, degradation_type=None, severity=0):
    """Evaluate model on test set with specified degradation."""
    model.eval()
    dataset = SkinLesionDataset(
        test_df,
        transform=eval_transform,
        degradation_type=degradation_type,
        severity=severity
    )
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)

    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1, all_preds, all_labels


# ── Run full robustness evaluation ──────────────────────────────
print("Running robustness evaluation... (this takes ~10-15 min)")

robustness_results = {}  # {model_name: {deg_type: [(severity, acc, f1), ...]}}

for model_name, model in trained_models.items():
    robustness_results[model_name] = {}
    print(f"\n  Evaluating: {model_name}")

    for deg_type in ['gaussian_noise', 'gaussian_blur', 'downsampling', 'jpeg_compression']:
        results = []
        for severity in range(6):  # 0 = clean, 1-5 = increasing degradation
            acc, f1, _, _ = evaluate_robustness(
                model, test_df,
                degradation_type=deg_type if severity > 0 else None,
                severity=severity
            )
            results.append((severity, acc, f1))
            print(f"    {deg_type} sev={severity}: acc={acc:.4f} f1={f1:.4f}")
        robustness_results[model_name][deg_type] = results

print("\nRobustness evaluation complete!")

## 9. Robustness Visualization — Main Paper Figures

These degradation curves are the **primary contribution figures** of your paper.

In [ ]:
# ── Degradation curves ──────────────────────────────────────────
MODEL_COLORS = {
    'EfficientNetB0 (CNN)':        '#e74c3c',
    'ViT-B/16 (Transformer)':      '#3498db',
    'EfficientFormer-L1 (Hybrid)': '#2ecc71',
}
MODEL_MARKERS = {
    'EfficientNetB0 (CNN)':        'o',
    'ViT-B/16 (Transformer)':      's',
    'EfficientFormer-L1 (Hybrid)': '^',
}

DEG_LABELS = {
    'gaussian_noise':    'Gaussian Noise',
    'gaussian_blur':     'Gaussian Blur',
    'downsampling':      'Resolution Downsampling',
    'jpeg_compression':  'JPEG Compression',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Robustness to Image Degradation — Weighted F1 Score\n(Trained on clean images, tested on degraded)',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, deg_type in enumerate(['gaussian_noise', 'gaussian_blur', 'downsampling', 'jpeg_compression']):
    ax = axes[idx]
    for model_name in trained_models:
        results = robustness_results[model_name][deg_type]
        severities = [r[0] for r in results]
        f1_scores  = [r[2] for r in results]
        ax.plot(severities, f1_scores,
                color=MODEL_COLORS[model_name],
                marker=MODEL_MARKERS[model_name],
                label=model_name, linewidth=2.5, markersize=7)

    ax.set_title(DEG_LABELS[deg_type], fontweight='bold', fontsize=12)
    ax.set_xlabel('Severity Level (0=Clean, 5=Worst)')
    ax.set_ylabel('Weighted F1 Score')
    ax.set_xticks(range(6))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('robustness_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: robustness_curves.png  ← Main contribution figure of your paper")

In [ ]:
# ── Robustness drop heatmap ─────────────────────────────────────
# Shows how much F1 DROPS from clean to worst degradation
drop_data = {}
for model_name in trained_models:
    drops = {}
    for deg_type in DEG_LABELS:
        results = robustness_results[model_name][deg_type]
        clean_f1 = results[0][2]  # severity 0
        worst_f1 = results[5][2]  # severity 5
        drops[DEG_LABELS[deg_type]] = round((clean_f1 - worst_f1) * 100, 1)
    drop_data[model_name] = drops

drop_df = pd.DataFrame(drop_data).T

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(drop_df, annot=True, fmt='.1f', cmap='Reds',
            ax=ax, cbar_kws={'label': 'F1 Drop (percentage points)'})
ax.set_title('F1 Score Drop: Clean → Worst Degradation (higher = less robust)',
             fontweight='bold')
ax.set_xlabel('Degradation Type')
ax.set_ylabel('Architecture')
plt.tight_layout()
plt.savefig('robustness_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: robustness_heatmap.png  ← Table alternative for paper")

In [ ]:
# ── Per-class robustness analysis ───────────────────────────────
# KEY SECONDARY FINDING: Does degradation hurt rare classes more?
# Compare clean vs severity-5 per class per model

print("Per-class F1 drop under worst degradation (severity 5)...\n")

per_class_results = {}

for model_name, model in trained_models.items():
    print(f"Model: {model_name}")
    per_class_results[model_name] = {}

    for deg_type in DEG_LABELS:
        # Clean
        _, _, preds_clean, labels_clean = evaluate_robustness(model, test_df)
        f1_clean = f1_score(labels_clean, preds_clean, average=None)

        # Worst
        _, _, preds_worst, labels_worst = evaluate_robustness(
            model, test_df, degradation_type=deg_type, severity=5
        )
        f1_worst = f1_score(labels_worst, preds_worst, average=None)

        drop = f1_clean - f1_worst
        per_class_results[model_name][deg_type] = {
            'clean': f1_clean, 'worst': f1_worst, 'drop': drop
        }

        class_names_short = list(CLASS_NAMES.keys())
        print(f"  {DEG_LABELS[deg_type]}:")
        for cls, d in zip(class_names_short, drop):
            print(f"    {cls}: {d:+.3f}")
    print()

In [ ]:
# ── Visualize per-class drops ───────────────────────────────────
# Focus on Gaussian noise (usually most revealing)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Per-Class F1 Drop under Gaussian Noise (Severity 5)\nDo rare classes suffer more?',
             fontsize=13, fontweight='bold')

class_labels = list(CLASS_NAMES.values())
# Approximate class frequency for coloring
class_freqs  = df['dx'].value_counts()[list(CLASS_NAMES.keys())].values

for i, (model_name, model) in enumerate(trained_models.items()):
    drops = per_class_results[model_name]['gaussian_noise']['drop']
    colors = plt.cm.RdYlGn_r(drops / max(drops + 1e-8))
    bars = axes[i].barh(class_labels, drops, color=colors, edgecolor='black', linewidth=0.5)
    axes[i].set_title(model_name.split('(')[0].strip(), fontweight='bold')
    axes[i].set_xlabel('F1 Drop (higher = less robust)')
    axes[i].axvline(0, color='black', linewidth=0.8)
    axes[i].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('per_class_robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: per_class_robustness.png  ← Secondary finding figure")

## 10. Summary Results Table

In [ ]:
# ── Build full results table ────────────────────────────────────
rows = []
for model_name in trained_models:
    for deg_type in DEG_LABELS:
        for severity, acc, f1 in robustness_results[model_name][deg_type]:
            rows.append({
                'Model': model_name,
                'Degradation': DEG_LABELS[deg_type],
                'Severity': severity,
                'Accuracy': round(acc, 4),
                'F1_Weighted': round(f1, 4),
            })

results_df = pd.DataFrame(rows)
results_df.to_csv('robustness_results.csv', index=False)
print("Saved: robustness_results.csv")

# Clean baseline comparison
print("\n=== CLEAN BASELINE (Severity 0) ===")
clean = results_df[results_df['Severity'] == 0].drop_duplicates('Model')
print(clean[['Model', 'Accuracy', 'F1_Weighted']].to_string(index=False))

print("\n=== WORST DEGRADATION AVERAGE (Severity 5) ===")
worst = results_df[results_df['Severity'] == 5].groupby('Model')[['Accuracy','F1_Weighted']].mean().round(4)
print(worst)

In [ ]:
# ── Download everything ─────────────────────────────────────────
import shutil
from google.colab import files

os.makedirs('paper_figures', exist_ok=True)

figure_files = [
    'sample_images.png',
    'class_distribution.png',
    'degradation_examples.png',
    'training_curves.png',
    'robustness_curves.png',
    'robustness_heatmap.png',
    'per_class_robustness.png',
    'robustness_results.csv',
]

for f in figure_files:
    if os.path.exists(f):
        shutil.copy(f, f'paper_figures/{f}')

shutil.make_archive('paper_figures', 'zip', 'paper_figures')
files.download('paper_figures.zip')
print("All figures downloaded!")

## 11. Paper Outline

```
TITLE:
  Robustness of CNN, Vision Transformer, and Hybrid Architectures
  to Image Quality Degradation in Dermatological Disease Classification

1. INTRODUCTION
   - Skin cancer kills X people/year, early detection critical
   - AI dermatology tools trained on clean images, deployed in messy reality
   - Gap: no study systematically compares CNN vs ViT vs Hybrid robustness
     under controlled degradation on skin lesion datasets
   - Our contribution: first systematic robustness benchmark on HAM10000

2. RELATED WORK
   - CNN in dermatology (cite 3-4 papers)
   - ViT in medical imaging (cite 2-3 papers)
   - Robustness in general vision (Bai et al. 2021, Paul & Chen 2022)
   - Gap: none apply this to skin lesion classification

3. DATASET
   - HAM10000 description
   - Figure 1: sample_images.png
   - Figure 2: class_distribution.png
   - Class imbalance handling (weighted loss)

4. METHODOLOGY
   4.1 Architectures (CNN / ViT / Hybrid — justify each choice)
   4.2 Training Protocol (identical for all — this is critical)
   4.3 Degradation Functions (Table 1: types + parameter ranges)
   4.4 Figure 3: degradation_examples.png
   4.5 Evaluation Metrics (Accuracy + Weighted F1)

5. RESULTS
   5.1 Baseline Clean Performance
       - Table 2: clean accuracy + F1 per model
       - Figure 4: training_curves.png
   5.2 Robustness Under Degradation
       - Figure 5: robustness_curves.png  ← MAIN RESULT
       - Figure 6: robustness_heatmap.png
   5.3 Per-Class Robustness
       - Figure 7: per_class_robustness.png
       - Do rare classes degrade faster? (novel secondary finding)

6. DISCUSSION
   - Which architecture is most robust and WHY
     (ViT global attention = less sensitive to local pixel corruption?)
   - Which degradation type is most damaging
   - Which disease classes are most vulnerable
   - Implications for deploying dermatology AI in low-resource settings
   - Limitations: single dataset, no adversarial attacks, no real-world
     degraded images (only simulated)

7. CONCLUSION
   - Recommended architecture for real-world deployment
   - Future work: test on real degraded clinical images,
     robustness-aware training (augment with degradations during training)

REFERENCES (aim for 20+)

---
SUBMISSION TARGETS:
  Tier 1 (aim first): PLOS ONE, IEEE Access, Scientific Reports
  Tier 2 (backup):    ResearchGate preprint, arXiv cs.CV
  Conference:         AMIA, IEEE EMBC, MICCAI workshop tracks
```